# Agentic Researcher: open-weight bulk research

This notebook runs a previously curated `research-queue/v1` with a local
OpenAI-compatible vLLM server and OpenCode. Commercial curators should create
and approve the queue before this notebook is used. Projects and batch state
are stored on Google Drive so an interrupted Colab session can resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/agentic-researcher')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
QUEUE = DRIVE_ROOT / 'research_queue.json'
WORK_ROOT = DRIVE_ROOT / 'projects'
STATE = DRIVE_ROOT / 'batch_state.json'
assert QUEUE.exists(), f'Upload the curated queue to {QUEUE}'


In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('AGENTIC_RESEARCHER_REPO', 'https://github.com/Erikiss/The-Agentic-Researcher.git')
REPO_REF = os.environ.get('AGENTIC_RESEARCHER_REF', 'main')
repo = Path('/content/The-Agentic-Researcher')
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(repo)], check=True)
!pip -q install uv
!uv pip install --system vllm
!npm install -g opencode-ai


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
MODEL = 'openai/gpt-oss-120b' if vram_gib >= 75 else 'openai/gpt-oss-20b'
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB)')
print(f'Model: {MODEL}')


In [ ]:
import json, os
from pathlib import Path

config_dir = Path.home() / '.config' / 'opencode'
config_dir.mkdir(parents=True, exist_ok=True)
config = {
    '$schema': 'https://opencode.ai/config.json',
    'autoupdate': False,
    'provider': {
        'local': {
            'npm': '@ai-sdk/openai-compatible',
            'name': 'Local Agentic Researcher',
            'options': {'baseURL': 'http://127.0.0.1:8000/v1', 'apiKey': 'local'},
            'models': {'default': {'name': MODEL}},
        }
    },
    'model': 'local/default',
    'permission': {
        'external_directory': 'deny',
        'question': 'deny',
        'doom_loop': 'deny',
        'bash': {
            '*': 'allow',
            'git push *': 'deny',
            'git commit *': 'deny',
        },
    },
}
(config_dir / 'opencode.json').write_text(json.dumps(config, indent=2), encoding='utf-8')


In [ ]:
import subprocess, sys, time, urllib.request

log_path = DRIVE_ROOT / 'vllm.log'
log_handle = log_path.open('ab')
server = subprocess.Popen([
    'vllm', 'serve', MODEL,
    '--served-model-name', 'default',
    '--host', '127.0.0.1',
    '--port', '8000',
    '--max-model-len', '32768',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'openai',
], stdout=log_handle, stderr=subprocess.STDOUT)

for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError(f'vLLM stopped early; inspect {log_path}')
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2)
        print('vLLM is ready')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'vLLM did not become ready; inspect {log_path}')


In [ ]:
import os, subprocess

repo = Path('/content/The-Agentic-Researcher')
env = os.environ.copy()
env['PYTHONPATH'] = str(repo)
command = [
    sys.executable, '-m', 'agentic_researcher', 'run-batch', str(QUEUE),
    '--work-root', str(WORK_ROOT),
    '--state', str(STATE),
    '--provider', 'opencode',
    '--command', 'opencode run --auto {prompt}',
    '--timeout', '7200',
]
print(' '.join(command))
subprocess.run(command, cwd=repo, env=env, check=True)


Re-running the final cell is safe. Completed task IDs are read from
`batch_state.json`; only pending or retryable tasks are executed.